# 03-01 문자 단위 토크나이저 만들기

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 문장을 문자로 쪼개기

In [ ]:
corpus = [
    "나는 학교에 간다",
    "오늘 날씨가 좋다",
    "학교 앞에 고양이가 있다",
    "고양이는 날씨가 좋으면 잔다",
]
print(list(corpus[0]))

## 2. 어휘집 만들기

In [ ]:
chars = sorted(set("".join(corpus)))
print(len(chars), chars)

In [ ]:
special = ["<pad>", "<unk>"]
itos = special + chars  # 번호 -> 토큰
stoi = {t: i for i, t in enumerate(itos)}  # 토큰 -> 번호
print(len(itos))
print({t: stoi[t] for t in ["<pad>", "<unk>", " ", "나", "학"]})

## 3. 인코딩과 디코딩

In [ ]:
def encode(text):
    return [stoi.get(ch, stoi["<unk>"]) for ch in text]

def decode(ids):
    return "".join(itos[i] for i in ids if i != stoi["<pad>"])

ids = encode("나는 학교에 간다")
print(ids)
print(decode(ids))
print(encode("나는 집에 간다"))  # 어휘집에 없는 문자 '집'
print(decode(encode("나는 집에 간다")))

## 4. 클래스로 묶기

In [ ]:
class CharTokenizer:
    def __init__(self, texts, special=("<pad>", "<unk>")):
        chars = sorted(set("".join(texts)))
        self.itos = list(special) + chars
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]

    def __len__(self):
        return len(self.itos)

    def encode(self, text):
        return [self.stoi.get(ch, self.unk_id) for ch in text]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids if i != self.pad_id)

tok = CharTokenizer(corpus)
print(len(tok), tok.encode("고양이"), tok.decode(tok.encode("고양이")))

## 5. 길이가 다른 문장을 한 배치로: 패딩

In [ ]:
def make_batch(texts, tok):
    encoded = [tok.encode(t) for t in texts]
    max_len = max(len(e) for e in encoded)
    ids = torch.full((len(texts), max_len), tok.pad_id)  # (batch, max_len)
    for i, e in enumerate(encoded):
        ids[i, : len(e)] = torch.tensor(e)
    mask = ids != tok.pad_id  # (batch, max_len), 진짜 토큰이면 True
    return ids, mask

ids, mask = make_batch(corpus, tok)
print(ids.shape)
print(ids)
print(mask.sum(dim=1))  # 문장마다 진짜 토큰 수

In [ ]:
plt.figure(figsize=(9, 2.6))
plt.imshow(ids, cmap="viridis")
for i in range(ids.shape[0]):
    for j in range(ids.shape[1]):
        plt.text(j, i, int(ids[i, j]), ha="center", va="center", color="white", fontsize=8)
plt.xlabel("position (seq_len)")
plt.ylabel("sentence (batch)")
plt.title("token ids, 0 = <pad>")
plt.show()

## 6. 임베딩 레이어에 넣기

In [ ]:
emb = nn.Embedding(len(tok), 8, padding_idx=tok.pad_id)  # 어휘집 크기 x 8
x = emb(ids)  # (batch=4, seq_len, 8)
print(ids.shape, "->", x.shape)
print(x[1, -1])  # <pad> 자리는 0 벡터예요